# Lab 1 — Kamilya Islamova

## Task and protocol
Predict below / typical / above relative to the author's usual engagement.
The released author split is retained: 71,395 training posts and 17,623 test posts.
Within train, GroupShuffleSplit(seed=42, test_size=0.2) reserves 659 authors
(14,548 posts) for validation; the remaining 2,636 authors (56,847 posts) fit models.
No validation author occurs in fitting data. TF-IDF is fitted only on fitting texts.
After selection, models are refitted on all training authors and evaluated once on test.
The previously reproduced official baseline is the test comparator.

## Method
1. Reproduce the official HistGradientBoosting baseline.
2. Add shifted rolling means of log1p reactions over 3, 5 and 10 preceding posts,
   a 5-post standard deviation, and recent mean minus historical median.
   Add text length per word, exclamation count and uppercase ratio.
3. Fit word unigram/bigram TF-IDF (40,000 features, min_df=5) with balanced
   logistic regression (C=1). Pseudonymous identity tokens become a generic token.
4. On validation, compare baseline/enriched tabular models, text weights
   {0, .15, .3, .5}, and typical-class multipliers {1, 1.2, 1.4, 1.6}: 32 candidates.
   The multiplier changes the decision rule, not calibrated probabilities.
5. Select 70% enriched tabular + 30% text, with typical multiplier 1.4.

Current-post reactions, residuals and labels are excluded from model inputs
through an explicit column whitelist. History follows the repository's chronological
sorting/shift convention and excludes the current row. As in the baseline,
historical reaction counts are crawl-time snapshots, not reconstructed as-of-post counts.

## Validation ablation
| Variant | Macro-F1 |
|---|---:|
| Official baseline | 0.5546 |
| Text only | 0.4466 |
| Added history/style | 0.5740 |
| Baseline + best class multiplier | 0.5922 |
| Added history/style + best class multiplier | 0.6112 |
| Added history/style + text, no multiplier | 0.5877 |
| Selected combination | 0.6176 |

The text-only model is weak, but provides complementary information. A large part
of the improvement also comes from the class decision rule; the gain should not
be attributed to text alone. History and style were added together, so this
experiment does not isolate their individual contributions.

## Held-out test
| Model | Macro-F1 | QWK | Accuracy |
|---|---:|---:|---:|
| Official baseline | 0.5410 | 0.4576 | 0.5465 |
| Selected solution | 0.6052 | 0.5240 | 0.6444 |

Macro-F1 improves by 0.0642 (about 11.9% relative). This is one held-out comparison,
not a statistical significance claim. Per-class F1: below 0.4850, typical 0.7085,
above 0.6220. Below does not improve (baseline 0.4859); the gain is concentrated
in typical and above. Predictions cover all 17,623 test posts and pass evaluate.py.

Confusion matrix (rows=true, columns=predicted; below, typical, above):
```
1784 1683   74
1900 7366 1225
 131 1253 2207
```

## Limitations
A single validation split and 32 candidates can overfit validation. The original
baseline also uses age at crawl and follower counts observed in 2026; these are
collection-time signals, not necessarily information available at publication.
The model is a retrospective dataset benchmark, not a proven live forecasting system.
No media files or external model APIs are used.

## Reproduce
From the repository root, with requirements.txt installed, run:
```
python improved.py
python evaluate.py --predictions lab1_submission/predictions.csv
```
Tested with Python 3.14.8 and scikit-learn 1.9.1. The script writes selection.json
before computing test performance, then metrics.json and predictions.csv.
The notebook uses the same script rather than duplicating the implementation.


In [ ]:
# Run the complete reproducible experiment (overwrites generated outputs).
import subprocess, sys
subprocess.run([sys.executable, "improved.py"], check=True)


In [ ]:
import json
from pathlib import Path
report = json.loads(Path("lab1_submission/metrics.json").read_text())
report["test"]


In [ ]:
subprocess.run([sys.executable, "evaluate.py", "--predictions",
                "lab1_submission/predictions.csv"], check=True)
